In [1]:
import optuna
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

In [2]:
!nvidia-smi

Sat Nov 15 00:36:51 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 531.29                 Driver Version: 531.29       CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                      TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce GTX 1650       WDDM | 00000000:01:00.0  On |                  N/A |
| N/A   46C    P8                2W /  N/A|    179MiB /  4096MiB |     21%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

# Criando o Dataset

In [7]:
# Settings
N_SAMPLES = 200_000
N_COLS = 50
N_INFORMATIVE = 45
N_CLASSES = 2
SEED = 42
TEST_SIZE = 0.2
EVAL_SIZE = 0.15

In [6]:
X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_COLS,
    n_informative=N_INFORMATIVE,
    n_classes=N_CLASSES,
    random_state=SEED
)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=TEST_SIZE, shuffle=True, stratify=y)
X_train, X_eval, y_train, y_eval = train_test_split(X_train, y_train, test_size=EVAL_SIZE, shuffle=True, stratify=y_train)

## LightGBM

### Sem usar a GPU

In [17]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbose": -1,
        "boosting_type": "gbdt",

        # GPU
        # "device": "gpu",
        # "gpu_platform_id": 0,
        # "gpu_device_id": 0,
        # "gpu_use_dp": False,  # usa float32, mais rápido
        # "max_bin": 255,  # binning menor → GPU mais rápida

        # Hiperparâmetros
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
    }
    train_set = lgb.Dataset(X_train, label=y_train)
    valid_set = lgb.Dataset(X_eval, label=y_eval)

    model = lgb.train(
        params,
        train_set,
        valid_sets=[valid_set],
        num_boost_round=2000,
    )
    preds = model.predict(X_eval)
    auc = roc_auc_score(y_eval, preds)
    return auc

In [18]:
%%time
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Melhores parâmetros:")
print(study.best_params)
print("Melhor AUC:", study.best_value)

[I 2025-11-15 00:41:11,747] A new study created in memory with name: no-name-d8abd47e-473e-42f8-93ef-0cec66edf55b
[I 2025-11-15 00:41:53,895] Trial 0 finished with value: 0.9836188253650655 and parameters: {'num_leaves': 101, 'max_depth': 8, 'learning_rate': 0.0014096979107941676, 'feature_fraction': 0.7780602770154474, 'bagging_fraction': 0.9969392675655863, 'bagging_freq': 3, 'min_data_in_leaf': 161, 'lambda_l1': 2.300819664707822e-05, 'lambda_l2': 0.015814034043977836}. Best is trial 0 with value: 0.9836188253650655.
[I 2025-11-15 00:42:05,188] Trial 1 finished with value: 0.8944053614558907 and parameters: {'num_leaves': 133, 'max_depth': 3, 'learning_rate': 0.0007693145077933955, 'feature_fraction': 0.6747508313930788, 'bagging_fraction': 0.8072064154319967, 'bagging_freq': 4, 'min_data_in_leaf': 85, 'lambda_l1': 0.16492920183231569, 'lambda_l2': 3.223531625056055e-07}. Best is trial 0 with value: 0.9836188253650655.
[I 2025-11-15 00:42:22,787] Trial 2 finished with value: 0.92398

Melhores parâmetros:
{'num_leaves': 235, 'max_depth': 11, 'learning_rate': 0.21584791768945155, 'feature_fraction': 0.8222319795178678, 'bagging_fraction': 0.7474834698167129, 'bagging_freq': 9, 'min_data_in_leaf': 113, 'lambda_l1': 1.6757852746562442e-05, 'lambda_l2': 0.002399258433364002}
Melhor AUC: 0.9948887844027713
CPU times: total: 5h 41min 26s
Wall time: 26min 37s


### Usando a GPU

In [21]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbose": -1,
        "boosting_type": "gbdt",

        # GPU
        "device": "gpu",
        "gpu_platform_id": 0,
        "gpu_device_id": 0,
        "gpu_use_dp": False,  # usa float32, mais rápido
        "max_bin": 255,  # binning menor → GPU mais rápida

        # Hiperparâmetros
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 200),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
    }
    train_set = lgb.Dataset(X_train, label=y_train)
    valid_set = lgb.Dataset(X_eval, label=y_eval)

    model = lgb.train(
        params,
        train_set,
        valid_sets=[valid_set],
        num_boost_round=2000,
    )
    preds = model.predict(X_eval)
    auc = roc_auc_score(y_eval, preds)
    return auc

In [22]:
%%time
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Melhores parâmetros:")
print(study.best_params)
print("Melhor AUC:", study.best_value)

[I 2025-11-15 01:12:55,218] A new study created in memory with name: no-name-8bbeeeca-5613-4e1a-9273-46a595ef9afa
[I 2025-11-15 01:15:31,832] Trial 0 finished with value: 0.9946341871646367 and parameters: {'num_leaves': 192, 'max_depth': 7, 'learning_rate': 0.05434598846558709, 'feature_fraction': 0.7933648727340608, 'bagging_fraction': 0.9400097384222641, 'bagging_freq': 4, 'min_data_in_leaf': 128, 'lambda_l1': 5.077662540406419e-06, 'lambda_l2': 7.79398185809333e-08}. Best is trial 0 with value: 0.9946341871646367.
[I 2025-11-15 01:16:13,078] Trial 1 finished with value: 0.9119604319975271 and parameters: {'num_leaves': 115, 'max_depth': 4, 'learning_rate': 0.0002623494157386185, 'feature_fraction': 0.6130264721248551, 'bagging_fraction': 0.5915570215028443, 'bagging_freq': 1, 'min_data_in_leaf': 43, 'lambda_l1': 0.007245823922247751, 'lambda_l2': 4.698308586034351e-06}. Best is trial 0 with value: 0.9946341871646367.
[I 2025-11-15 01:18:36,993] Trial 2 finished with value: 0.993626

Melhores parâmetros:
{'num_leaves': 90, 'max_depth': 11, 'learning_rate': 0.0916882483581474, 'feature_fraction': 0.97379754331081, 'bagging_fraction': 0.6633672901765577, 'bagging_freq': 8, 'min_data_in_leaf': 106, 'lambda_l1': 1.0786844955870159e-08, 'lambda_l2': 5.026652820082367e-07}
Melhor AUC: 0.9949621628795795
CPU times: total: 23h 45min 56s
Wall time: 9h 49min 53s
